# Machine Learning – Practical 6: Artificial Neural Network (Breast Cancer Classification)

**Aim:** To build an ANN that classifies tumours as malignant or benign and study the effect of hyperparameters.

**Objective:**
- Build a feed-forward ANN with Keras
- Compare different epochs, batch sizes and activation functions
- Evaluate the final model using a classification report and confusion matrix

**Theory:** An ANN consists of layers of neurons that apply weights, biases and activation functions. Weights are learned by backpropagation using an optimiser (Adam) to minimise the loss (binary cross-entropy). Epochs = full passes over data, batch size = samples per weight update, activation = non-linearity (ReLU, tanh, sigmoid).

## 1. Import Libraries

**Aim:** Import required libraries.

**Objective:** Load TensorFlow/Keras, scikit-learn and plotting tools.

**Theory:** Keras (inside TensorFlow) offers a high-level API for building neural networks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

## 2. Load Dataset

**Aim:** Load the Breast Cancer Wisconsin dataset.

**Objective:** Get features and labels.

**Theory:** The dataset has 569 samples and 30 numeric features describing cell nuclei; classes are malignant (0) and benign (1).

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset: Breast Cancer Wisconsin")
print("Number of samples:", X.shape[0])
print("Number of input features:", X.shape[1])
print("Class names:", data.target_names)

## 3. Split & Standardise Data

**Aim:** Prepare data for the neural network.

**Objective:** 80/20 stratified split, then standardise features.

**Theory:** Neural networks train faster and more stably when inputs are scaled. The scaler is fitted on training data only to avoid leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Data standardization completed!")

## 4. Define ANN Architecture

**Aim:** Create a reusable model builder.

**Objective:** Build 30 → 16 → 8 → 1 network with a sigmoid output.

**Theory:** Two hidden layers learn complex patterns; the sigmoid output gives a probability for binary classification. Adam optimiser + binary cross-entropy loss are standard for this task.

In [ ]:
def build_ann(hidden_activation="relu"):

    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(16, activation=hidden_activation),
        Dense(8, activation=hidden_activation),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("ANN architecture defined successfully!")

## 5. Evaluation Function

**Aim:** Evaluate any trained model consistently.

**Objective:** Compute accuracy, precision, recall, F1 and test loss.

**Theory:** Predicted probabilities ≥ 0.5 are classified as benign (1), otherwise malignant (0).

In [ ]:
def evaluate_model(model, experiment, epochs, batch_size):

    probabilities = model.predict(
        X_test, verbose=0
    ).ravel()

    y_pred = (probabilities >= 0.5).astype(int)

    return {
        "Experiment": experiment,
        "Epochs": epochs,
        "Batch Size": batch_size,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "Test Loss": model.evaluate(
            X_test, y_test, verbose=0
        )[0]
    }

results = []

print("Evaluation function created successfully!")

## 6. Experiment 1 – Number of Epochs

**Aim:** Study the effect of training length.

**Objective:** Train with 20, 50 and 100 epochs.

**Theory:** Too few epochs underfit; too many can overfit. Comparing test metrics shows the best trade-off.

In [ ]:
epoch_values = [20, 50, 100]

for epochs in epoch_values:

    tf.keras.backend.clear_session()

    model = build_ann("relu")

    model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=32,
        verbose=0
    )

    results.append(
        evaluate_model(
            model,
            "Epoch Comparison",
            epochs,
            32
        )
    )

print("Epoch comparison completed.")

## 7. Experiment 2 – Batch Size

**Aim:** Study the effect of batch size.

**Objective:** Train with batch sizes 16, 32 and 64.

**Theory:** Small batches give noisier but more frequent updates; large batches are smoother and faster per epoch.

In [ ]:
batch_values = [16, 32, 64]

for batch_size in batch_values:

    tf.keras.backend.clear_session()

    model = build_ann("relu")

    model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=batch_size,
        verbose=0
    )

    results.append(
        evaluate_model(
            model,
            "Batch Size Comparison",
            50,
            batch_size
        )
    )

print("Batch-size comparison completed.")

## 8. Experiment 3 – Activation Functions

**Aim:** Study the effect of hidden-layer activations.

**Objective:** Compare ReLU, tanh and sigmoid.

**Theory:** ReLU is fast and avoids vanishing gradients; tanh is zero-centred; sigmoid can saturate and learn slower in hidden layers.

In [ ]:
activation_values = ["relu", "tanh", "sigmoid"]

for activation in activation_values:

    tf.keras.backend.clear_session()

    model = build_ann(activation)

    model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=32,
        verbose=0
    )

    results.append(
        evaluate_model(
            model,
            f"Activation: {activation}",
            50,
            32
        )
    )

print("Activation-function comparison completed.")

## 9. Results Comparison

**Aim:** Compare all experiments.

**Objective:** Tabulate results, save to CSV and plot accuracy and loss.

**Theory:** Bar charts make it easy to spot which hyperparameters give the best accuracy and lowest loss.

In [ ]:
results_df = pd.DataFrame(results)

print("ANN Performance Comparison:")
print(results_df.round(4).to_string(index=False))

results_df.to_csv(
    "ANN_Performance_Comparison.csv",
    index=False
)

print("\nComparison results saved successfully!")

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    range(len(results_df)),
    results_df["Accuracy"]
)

plt.xticks(
    range(len(results_df)),
    [
        f"{row['Experiment']}\n"
        f"E={row['Epochs']}, B={row['Batch Size']}"
        for _, row in results_df.iterrows()
    ],
    rotation=45,
    ha="right"
)

plt.ylabel("Test Accuracy")
plt.title("ANN Performance Comparison")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    range(len(results_df)),
    results_df["Test Loss"]
)

plt.xticks(
    range(len(results_df)),
    results_df["Experiment"],
    rotation=45,
    ha="right"
)

plt.ylabel("Test Loss")
plt.title("Comparison of ANN Test Loss")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Final Model Training

**Aim:** Train the final ANN with validation monitoring.

**Objective:** Train for 50 epochs using a 20% validation split.

**Theory:** Validation data is held out from training to monitor overfitting during learning.

In [ ]:
tf.keras.backend.clear_session()

final_model = build_ann("relu")

history = final_model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.20,
    verbose=0
)

print("Final ANN model trained successfully!")

## 11. Final Evaluation

**Aim:** Evaluate the final model on test data.

**Objective:** Print test loss, accuracy, classification report and confusion matrix.

**Theory:** The confusion matrix shows how many malignant and benign tumours were correctly or wrongly classified.

In [ ]:
test_loss, test_accuracy = final_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

y_probability = final_model.predict(
    X_test,
    verbose=0
).ravel()

y_pred = (y_probability >= 0.5).astype(int)

print("\nFinal ANN Model Results")
print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 4))

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    target_names=data.target_names,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 12. Learning Curve & Model Summary

**Aim:** Inspect training behaviour and architecture.

**Objective:** Plot training vs validation accuracy and print the model summary.

**Theory:** If training and validation accuracy stay close, the model generalises well.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN Training and Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("ANN Model Summary:")
final_model.summary()

print("\nExperiment completed.")

## Conclusion

The ANN reached about 95–97% test accuracy across all settings. The final model scored 95.6% accuracy (40/42 malignant and 69/72 benign correct). Hyperparameter changes had only small effects; sigmoid hidden layers gave the highest test loss.